In [ ]:
from pathlib import Path
import sys
code=Path('/kaggle/working/comparison_code');code.mkdir(exist_ok=True)
sources={'compare_experiments.py': '"""Paired descriptive comparison of frozen predictions; never fits thresholds."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import roc_auc_score, average_precision_score\n\n\ndef metrics(frame):\n    y = frame.y_true.to_numpy(dtype=int)\n    p = frame.probability.to_numpy(dtype=float)\n    pred = frame.prediction.to_numpy(dtype=int)\n    return {\n        \'auroc\': float(roc_auc_score(y, p)),\n        \'average_precision\': float(average_precision_score(y, p)),\n        \'accuracy\': float(np.mean(y == pred)),\n        \'sensitivity\': float(np.mean(pred[y == 1] == 1)),\n        \'specificity\': float(np.mean(pred[y == 0] == 0)),\n        \'brier_score\': float(np.mean((y - p) ** 2)),\n    }\n\n\ndef compare(baseline_dir, followup_dir, replicates=1000):\n    if replicates < 1:\n        raise ValueError(\'replicates must be positive\')\n    frames = []\n    for directory in (baseline_dir, followup_dir):\n        frame = pd.read_csv(Path(directory) / \'test_predictions.csv\')\n        required = {\'sha256\', \'patient\', \'y_true\', \'probability\', \'prediction\'}\n        if not required.issubset(frame.columns) or frame.sha256.duplicated().any():\n            raise ValueError(\'Missing fields or duplicate image hashes\')\n        if set(frame.y_true) != {0, 1} or not set(frame.prediction).issubset({0, 1}):\n            raise ValueError(\'Expected binary labels with both classes represented\')\n        if not np.isfinite(frame.probability).all() or not frame.probability.between(0, 1).all():\n            raise ValueError(\'Invalid probabilities\')\n        if frame.patient.isna().any() or frame.sha256.isna().any():\n            raise ValueError(\'Missing pairing or cluster identifiers\')\n        frames.append(frame.sort_values(\'sha256\').reset_index(drop=True))\n    a, b = frames\n    for column in (\'sha256\', \'patient\', \'y_true\'):\n        if not a[column].equals(b[column]):\n            raise ValueError(\'Test cohorts or labels differ: \' + column)\n    ma, mb = metrics(a), metrics(b)\n    groups = [np.flatnonzero(a.patient.to_numpy() == group) for group in a.patient.unique()]\n    rng = np.random.default_rng(42)\n    differences = []\n    for _ in range(replicates):\n        idx = np.concatenate([groups[i] for i in rng.integers(0, len(groups), len(groups))])\n        if a.iloc[idx].y_true.nunique() != 2:\n            continue\n        sa, sb = metrics(a.iloc[idx]), metrics(b.iloc[idx])\n        differences.append({key: sb[key] - sa[key] for key in sa})\n    if not differences:\n        raise ValueError(\'No bootstrap sample contained both classes\')\n    return {\n        \'status\': \'Exploratory paired comparison; original test results already inspected\',\n        \'direction\': \'followup minus baseline; lower is better for Brier score only\',\n        \'images\': len(a), \'filename_proxy_groups\': len(groups),\n        \'baseline\': ma, \'followup\': mb,\n        \'difference\': {key: mb[key] - ma[key] for key in ma},\n        \'paired_cluster_bootstrap_95ci\': {\n            key: np.percentile([d[key] for d in differences], [2.5, 97.5]).tolist()\n            for key in ma},\n        \'bootstrap_replicates\': len(differences),\n        \'limitations\': \'Filename groups are proxies. Intervals describe test sampling, not training variation, model selection uncertainty or external generalization. No multiplicity adjustment. Thresholds remain those saved by each experiment.\'\n    }\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'baseline\', type=Path)\n    parser.add_argument(\'followup\', type=Path)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--replicates\', type=int, default=1000)\n    args = parser.parse_args()\n    report = compare(args.baseline, args.followup, args.replicates)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2))\n', 'analyze_benchmark_comparison.py': '"""Descriptive paired uncertainty for already-frozen source test predictions."""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom compare_experiments import compare, metrics\n\n\ndef analyze(previous, benchmark, output, replicates=1000):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    comparison=compare(previous,benchmark,replicates)\n    frame=pd.read_csv(Path(benchmark)/\'test_predictions.csv\').sort_values(\'sha256\').reset_index(drop=True)\n    groups=[np.flatnonzero(frame.patient.to_numpy()==g) for g in frame.patient.unique()]\n    rng=np.random.default_rng(42); samples=[]\n    for _ in range(replicates):\n        indices=np.concatenate([groups[i] for i in rng.integers(0,len(groups),len(groups))])\n        sample=frame.iloc[indices]\n        if sample.y_true.nunique()==2: samples.append(metrics(sample))\n    if not samples: raise ValueError(\'No usable bootstrap samples\')\n    intervals={name:np.percentile([m[name] for m in samples],[2.5,97.5]).tolist() for name in samples[0]}\n    report={\'status\':\'Exploratory; the original source test was previously inspected\',\n            \'benchmark_metrics\':metrics(frame),\'benchmark_cluster_bootstrap_95ci\':intervals,\n            \'bootstrap_replicates\':len(samples),\'paired_comparison\':comparison,\n            \'interpretation\':\'Conditional test-sampling uncertainty only. Does not capture training, model selection, proxy-group error or external generalization. No threshold refitting or model selection occurs.\'}\n    (output/\'comparison.json\').write_text(json.dumps(report,indent=2))\n    print(\'COMPARISON_DONE\',json.dumps(report),flush=True)\n    return report\n\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'previous\',type=Path);parser.add_argument(\'benchmark\',type=Path)\n    parser.add_argument(\'--output\',type=Path,required=True)\n    args=parser.parse_args();analyze(args.previous,args.benchmark,args.output)\n'}
for name,text in sources.items(): (code/name).write_text(text)
sys.path.insert(0,str(code))
from analyze_benchmark_comparison import analyze
root=Path('/kaggle/input')
previous=[p.parent for p in root.rglob('test_predictions.csv') if p.parent.name=='pneumonia_ensemble_outputs']
benchmark=[p.parent for p in root.rglob('test_predictions.csv') if p.parent.name=='pneumonia_benchmark_outputs']
if len(previous)!=1 or len(benchmark)!=1: raise ValueError('Attach the completed follow-up and 224-pixel benchmark outputs')
analyze(previous[0],benchmark[0],Path('/kaggle/working/comparison_outputs'))
